# Lesson 03 — Shi-Tomasi Corner Detector

## What is Shi-Tomasi?

Shi-Tomasi (1994) is a refinement of the Harris corner detector.  
Harris scores a corner using: **R = det(M) - k * trace(M)²**  
Shi-Tomasi simplifies this — it just looks at the **minimum eigenvalue** of the matrix M:

> A point is a corner if **min(λ1, λ2) > threshold**

This makes it more stable and is the default corner detector inside `cv2.calcOpticalFlowPyrLK`.

---

## When to use it
- Tracking points across frames (optical flow)
- Anywhere you need strong, stable corners
- When Harris gives too many weak/noisy corners

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

## 1. Load image and convert to grayscale

In [ ]:
img = cv2.imread('your_image.jpg')         # replace with your image path
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gray = np.float32(gray)

plt.imshow(img_rgb)
plt.title('Original Image')
plt.axis('off')
plt.show()

## 2. Detect corners with goodFeaturesToTrack

```python
cv2.goodFeaturesToTrack(image, maxCorners, qualityLevel, minDistance)
```

| Parameter | Meaning |
|---|---|
| `maxCorners` | max number of corners to return |
| `qualityLevel` | minimum quality (0–1), corners below this * best_corner_score are rejected |
| `minDistance` | minimum pixel distance between corners |
| `useHarrisDetector` | False = Shi-Tomasi (default), True = Harris |

In [ ]:
corners = cv2.goodFeaturesToTrack(
    gray,
    maxCorners=100,
    qualityLevel=0.01,
    minDistance=10,
    useHarrisDetector=False    # Shi-Tomasi
)

corners = np.int32(corners)
print(f'Corners detected: {len(corners)}')

## 3. Draw detected corners

In [ ]:
output = img_rgb.copy()

for corner in corners:
    x, y = corner.ravel()
    cv2.circle(output, (x, y), 4, (0, 255, 0), -1)

plt.figure(figsize=(10, 6))
plt.imshow(output)
plt.title(f'Shi-Tomasi Corners ({len(corners)} detected)')
plt.axis('off')
plt.show()

## 4. Effect of qualityLevel parameter

In [ ]:
quality_levels = [0.001, 0.01, 0.05, 0.1]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, ql in zip(axes, quality_levels):
    c = cv2.goodFeaturesToTrack(gray, maxCorners=200, qualityLevel=ql, minDistance=10)
    vis = img_rgb.copy()
    if c is not None:
        c = np.int32(c)
        for pt in c:
            x, y = pt.ravel()
            cv2.circle(vis, (x, y), 4, (0, 255, 0), -1)
    ax.imshow(vis)
    ax.set_title(f'qualityLevel={ql}\n({len(c) if c is not None else 0} corners)')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 5. Shi-Tomasi vs Harris — side by side comparison

In [ ]:
# Shi-Tomasi
shi_corners = cv2.goodFeaturesToTrack(gray, 100, 0.01, 10, useHarrisDetector=False)
shi_corners = np.int32(shi_corners)

# Harris (via goodFeaturesToTrack)
harris_corners = cv2.goodFeaturesToTrack(gray, 100, 0.01, 10, useHarrisDetector=True, k=0.04)
harris_corners = np.int32(harris_corners)

shi_vis = img_rgb.copy()
harris_vis = img_rgb.copy()

for pt in shi_corners:
    cv2.circle(shi_vis, tuple(pt.ravel()), 4, (0, 255, 0), -1)

for pt in harris_corners:
    cv2.circle(harris_vis, tuple(pt.ravel()), 4, (255, 0, 0), -1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.imshow(shi_vis)
ax1.set_title(f'Shi-Tomasi ({len(shi_corners)} corners)')
ax1.axis('off')
ax2.imshow(harris_vis)
ax2.set_title(f'Harris ({len(harris_corners)} corners)')
ax2.axis('off')
plt.tight_layout()
plt.show()

## 6. With subpixel accuracy

Corner positions are integer by default.  
`cv2.cornerSubPix` refines them to subpixel precision — important for camera calibration and optical flow.

In [ ]:
corners_float = cv2.goodFeaturesToTrack(gray, 100, 0.01, 10)

criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

corners_subpix = cv2.cornerSubPix(
    gray,
    corners_float,
    winSize=(5, 5),
    zeroZone=(-1, -1),
    criteria=criteria
)

print('Before subpix:', corners_float[0])
print('After subpix: ', corners_subpix[0])

## Summary

| | Harris | Shi-Tomasi |
|---|---|---|
| Score | det(M) - k·trace(M)² | min(λ1, λ2) |
| Stability | Good | Better |
| Speed | Fast | Fast |
| Used in | General detection | Optical flow, tracking |
| OpenCV function | `cornerHarris` | `goodFeaturesToTrack` |

**Key takeaway:** When you need corners to track across video frames, always use Shi-Tomasi.  
It selects corners that survive motion better than Harris.

---

**Next:** Lesson 04 — FAST Detector